In [1]:
#image_path="570216.png"
#image_path="1920x1080-full-hd-nature-landscape.jpg"
#image_path="testpattern-hd-1080.jpg"
#image_path="ISOLDE_1600x1300.png"
image_path="test_1300x1600.png"


In [2]:

from PIL import Image
import numpy as np

import numpy as np
from PIL import Image

def image_to_tensor(filepath, dtype=np.float32, normalize=True, grayscale=False):
    """
    Convert an image to tensor format (1, C, H, W).

    Parameters
    ----------
    filepath : str
        Path to image.
    dtype : numpy dtype
        Output dtype (e.g., np.float32 or np.int32).
    normalize : bool
        If True and dtype is float32, scale to [0, 1].
    grayscale : bool
        If True -> C=1, else C=3 (RGB).

    Returns
    -------
    img_tensor : np.ndarray
        Shape (1, C, H, W)
    """

    # Select mode
    mode = 'L' if grayscale else 'RGB'

    # Open and convert image
    image = Image.open(filepath).convert(mode)

    # Convert to NumPy array
    img_np = np.array(image, dtype=dtype)

    # Normalize or clip
    if dtype == np.float32:
        if normalize:
            img_np /= 255.0
    elif dtype == np.int32:
        img_np = np.clip(img_np, 0, 255)

    # Ensure channel dimension exists
    if grayscale:
        # (H, W) -> (1, H, W)
        img_np = np.expand_dims(img_np, axis=0)
    else:
        # (H, W, 3) -> (3, H, W)
        img_np = np.transpose(img_np, (2, 0, 1))

    # Add batch dimension -> (1, C, H, W)
    img_tensor = np.expand_dims(img_np, axis=0)

    return img_tensor
    
def tensor_to_image(img_tensor, filepath, denormalize=True):
    img_np = np.squeeze(img_tensor, axis=0)
    num_channels = img_np.shape[0]
    
    if num_channels == 1:
        img_np = np.squeeze(img_np, axis=0)
        mode = 'L'
    elif num_channels == 3:
        img_np = np.transpose(img_np, (1, 2, 0))
        mode = 'RGB'
    else:
        raise ValueError(f"Unsupported number of channels: {num_channels}")
    
    if img_np.dtype in [np.float32, np.float64]:
        if denormalize:
            img_np = img_np * 255.0
        img_np = np.clip(img_np, 0, 255).astype(np.uint8)
    else:
        img_np = np.clip(img_np, 0, 255).astype(np.uint8)
    
    image = Image.fromarray(img_np, mode=mode)
    image.save(filepath)
    print(f"Image saved to: {filepath}")

In [3]:
tensor= image_to_tensor(image_path,grayscale=True)
print(f"tensor shape{tensor.shape}")
tensor_to_image(tensor,"grey_1600x1300.jpg")

tensor shape(1, 1, 1300, 1600)
Image saved to: grey_1600x1300.jpg


In [ ]:

def create_kernel(kernel_size=3):

    channels = 3
    

    # Luminance coefficients for RGB
    luminance = np.array([0.2989, 0.5870, 0.1140], dtype=np.float32)

    # Create empty kernel: shape [1, channels, kernel_size, kernel_size]
    mean_kernel = np.zeros((1, channels, kernel_size, kernel_size), dtype=np.float32)

    # Fill each channel with its luminance spread over the kernel
    for c in range(channels):
        mean_kernel[0, c, :, :] = luminance[c] / (kernel_size ** 2)

    mean_kernel_q =  pow(2, 8)*mean_kernel
    mean_kernel_q = mean_kernel_q.astype(np.int32)

    return mean_kernel,mean_kernel_q
import torch
import torch.nn as nn
import torchvision.transforms as T
from PIL import Image
import torch.nn.functional as F

def exec_conv2d(np_x,np_w,stride=2,padding=0):
    torch_type= torch.float32
    match np_x.dtype:            
        case np.int32:
            torch_type = torch.int32
    torch_x = torch.from_numpy(np_x).to(torch_type)
    torch_w = torch.from_numpy(np_w).to(torch_type)
#Apply convolution
    output = F.conv2d(
        torch_x,
        torch_w,
        bias=None,
        stride=stride,
        padding=padding
    )
    return output

def serialize(y,x,w,sufix=""):
    np.save(f"y{sufix}",y)
    np.save(f"x{sufix}",x)
    np.save(f"w{sufix}",w)


In [18]:
import torch
import torch.nn as nn
import torchvision.transforms as T
from PIL import Image
import torch.nn.functional as F

kernel_size=5
stride=5
mean_kernel,mean_kernel_q = create_kernel(kernel_size=kernel_size)
print(f"kernel size:{mean_kernel_q.shape}")
np_x= image_to_tensor(image_path)
np_x_q= image_to_tensor(image_path,dtype=np.int32)
output = exec_conv2d(np_x,mean_kernel,stride=stride)
print(f"float shape{output.shape}")
#output =output.clamp(0, 1)
serialize(y=output.numpy(),x=np_x,w=mean_kernel)
#save jpg
output = T.ToPILImage()(output.squeeze(0))
output.save("output_downsampled.jpg")

#
output = exec_conv2d(np_x_q,mean_kernel_q,stride=stride)
#de-scale the kernel values
output =output/(pow(2, 8))
output = output.to(torch.int32)
print(f"int32 size:{output.shape}")
serialize(y=output.numpy(),x=np_x_q,w=mean_kernel_q,sufix="_int32")
np_x_q.tofile("x_int32.bin")
#save jpg
output = T.ToPILImage()(output.squeeze(0).to(torch.uint8))
output.save("output_downsampled_q.jpg")


kernel size:(1, 3, 5, 5)
float shapetorch.Size([1, 1, 260, 320])
int32 size:torch.Size([1, 1, 260, 320])


# Test

In [4]:
fname = "y_int32.npy"
np_array =np.load(fname)
print(f"{fname} -> shape: {np_array.shape}, dtype={np_array.dtype}")
print(np_array)

y_int32.npy -> shape: (1, 1, 539, 959), dtype=int32
[[[[241 241 199 ... 224 242 242]
   [241 241 199 ... 225 242 242]
   [241 241 199 ... 225 242 242]
   ...
   [241 241 199 ... 224 242 242]
   [241 241 199 ... 224 242 242]
   [241 241 199 ... 224 242 242]]]]


# chunked convolution

In [5]:
import torch

def find_mismatch_line(a, b):
    """
    Traverse two tensors along dim=2 (height) and return the first mismatch line.
    Assumes both tensors have the same shape along all other dimensions.

    Args:
        a (torch.Tensor): First tensor of shape [batch, channels, height, width]
        b (torch.Tensor): Second tensor of shape [batch, channels, height, width]
        
    Returns:
        int: The first line index where the mismatch is found. None if no mismatch.
    """
    # Check that tensors have the same shape
    if a.shape != b.shape:
        raise ValueError("Tensors must have the same shape")

    # Traverse along dim=2 (height)
    for line in range(a.shape[2]):  # a.shape[2] is the height (dim=2)
        if not torch.allclose(a[:, :, line, :], b[:, :, line, :], atol=1e-6):  # Set a tolerance value
            print(f"Mismatch found at line {line}")
            return line
    print("No mismatch found")
    return None

# Example usage
a = torch.rand((1, 3, 5, 4))  # Example tensor 1 with shape [batch, channels, height, width]
b = a.clone()  # Example tensor 2 (same as a for now)
#b[0, 0, 2, 2] = 0.99  # Introduce a slight mismatch

# Find the first mismatch line
mismatch_line = find_mismatch_line(a, b)
if mismatch_line is not None:
    print(f"First mismatch found at line: {mismatch_line}")
else:
    print("Tensors are identical along dim=2.")


No mismatch found
Tensors are identical along dim=2.


In [6]:
import torch
import numpy as np
import torch.nn.functional as F

def compute_output_dim(input_size, kernel_size, stride=2, padding=1):
    return (input_size + 2 * padding - kernel_size) // stride + 1
def save_img(fname, tensor):
    output = T.ToPILImage()(tensor.squeeze(0))
    output.save(f"{fname}.jpg")


def chunked_convolution(image_tensor, kernel):
   
    _, _, H, _ = image_tensor.shape
    
    n= 319 #239 #319 #479
    chunk_height = 2*n+3


    output_chunks = []
    start_h = 0
    end_h =  chunk_height
    chunk_cnt =0
    while end_h < H:

        if( chunk_cnt):
            start_h =  end_h-1
            end_h   =  min(start_h+chunk_height, H)
        else:
            start_h =  0
            end_h   =  chunk_height

        chunk = image_tensor[:, :, start_h:end_h, :]

    
        output_chunk = exec_conv2d(chunk,kernel,stride=2,padding=0)
        save_img(f"out_chunk_{chunk_cnt}", output_chunk)
        print(f"chunk {chunk_cnt}: input [{start_h}:{end_h}] -> {end_h-start_h} => output={output_chunk.shape}")
        
        output_chunks.append(output_chunk)        

        chunk_cnt +=1

    # Concatenate outputs
    output = torch.cat(output_chunks, dim=2)
    return output

from numpy.testing import assert_allclose 

mean_kernel,mean_kernel_q = create_kernel()
np_x= image_to_tensor(image_path)

output = chunked_convolution(np_x,mean_kernel)
print(output.shape)
save_img("output_chuncked_convolution",output)
#
expected = exec_conv2d(np_x,mean_kernel,stride=2,padding=0)
assert_allclose(expected, output,rtol=1e-6,atol=1e-7)
print(f"expected:{expected.shape}")
diff = (expected - output).abs()
diff_max = diff.max().item()
print(f"Max absolute difference: {diff_max}")

save_img("diff_visual", diff)  # You may need to normalize to [0,1]


chunk 0: input [0:641] -> 641 => output=torch.Size([1, 1, 320, 959])
chunk 1: input [640:1080] -> 440 => output=torch.Size([1, 1, 219, 959])
torch.Size([1, 1, 539, 959])
expected:torch.Size([1, 1, 539, 959])
Max absolute difference: 0.0


In [7]:
_,_,W_expected,_ = expected.shape
last_row =539
first_row =0
mismatch_line = find_mismatch_line(output[:,:,first_row:last_row,:], expected[:,:,first_row:last_row,:])
if mismatch_line is not None:
    print(f"First mismatch found at line: {mismatch_line}")
else:
    print("Tensors are identical along dim=2.")

No mismatch found
Tensors are identical along dim=2.
